# CRGCN trên BPATMP (REES46)

Notebook chạy CRGCN (https://github.com/MingshiYan/CRGCN) trên dataset BPATMP với:
- Hyperparams lấy từ `training.yaml` (đã auto-downsize cho phù hợp Colab GPU).
- Evaluation theo `evaluator.py` (full-rank, mask exclusion bằng `train_mask_purchase_only.pkl`, HR@k & NDCG@k).
- Lưu checkpoint lên Weights & Biases.

## 1. Cài đặt môi trường + clone CRGCN

In [1]:
!pip install -q loguru tensorboard pyyaml
!pip install -q torch_geometric

import torch
TORCH = torch.__version__.split('+')[0]
CUDA = ('cu' + torch.version.cuda.replace('.', '')) if torch.version.cuda else 'cpu'
WHL = f'https://data.pyg.org/whl/torch-{TORCH}+{CUDA}.html'
print('Installing torch_scatter/torch_sparse from:', WHL)
!pip install -q torch_scatter torch_sparse -f {WHL}

import torch_scatter, torch_sparse
print('torch:', torch.__version__, '| torch_scatter:', torch_scatter.__version__, '| torch_sparse:', torch_sparse.__version__)

import os, sys
if not os.path.isdir('/content/CRGCN'):
    !git clone --depth 1 https://github.com/MingshiYan/CRGCN.git /content/CRGCN
sys.path.insert(0, '/content/CRGCN')
print('CRGCN files:', os.listdir('/content/CRGCN'))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 7.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 83.9 MB/s eta 0:00:00
Installing torch_scatter/torch_sparse from: https://data.pyg.org/whl/torch-2.10.0+cu128.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 139.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 31.8 MB/s eta 0:00:00
torch: 2.10.0+cu128 | torch_scatter: 2.1.2+pt210cu128 | torch_sparse: 0.6.18+pt210cu128
Cloning into '/content/CRGCN'...
remote: Enumerating objects: 14, done.
remote: Counting objects: 100% (14/14), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 14 (delta 0), reused 9 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (14/14), 54.28 MiB | 15.40 MiB/s, done.
CRGCN files: ['.gitignore', '.git', 'data_set.py', 'CRGCN.sh', 'main.py', 'metrics.py', 'requirements.txt', 'README.md', '

## 2. Tải dataset BPATMP từ Hugging Face

In [2]:
import os
if not os.path.isdir('/content/data') or not os.path.isfile('/content/data/purchase_train_src.npy'):
    !pip install -q huggingface_hub hf_transfer
    os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
    from huggingface_hub import snapshot_download
    snapshot_download(
        repo_id='nguyenmaiductrong/rees46-bpatmp-temporal',
        repo_type='dataset',
        local_dir='/content/data',
    )
print('data:', sorted(os.listdir('/content/data'))[:20])

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 90.5 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 167 files:   0%|          | 0/167 [00:00<?, ?it/s]

data: ['.cache', '.gitattributes', 'artifacts_manifest.json', 'baseline_contract.md', 'candidate_item_idx.npy', 'cart_train_dst.npy', 'cart_train_src.npy', 'cart_train_ts.npy', 'data_card.md', 'graph', 'node_counts.json', 'node_mappings', 'purchase_train_dst.npy', 'purchase_train_src.npy', 'purchase_train_ts.npy', 'split_manifest.json', 'test_ground_truth.pkl', 'test_product_idx.npy', 'test_timestamp.npy', 'test_user_idx.npy']


## 3. Load `training.yaml`

In [3]:
import yaml, io, os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

YAML_TEXT = """
data:
  data_dir: /content/data/
  node_counts:
    product: 29892
    user: 203063
model:
  embed_dim: 64
  n_layers: 1
  dropout: 0.4
training:
  batch_size: 2048
  eval_batch_size: 1024
  epochs: 40
  lr: 1.0e-03
  l2_lambda: 1.0e-04
  seed: 42
  eval_every: 1
  patience: 5
  num_neg: 1
  device: cuda
  save_dir: /content/checkpoints-crgcn
  max_view_triplets: 3000000
  max_cart_triplets: 3868050
  max_purchase_triplets: 0
evaluation:
  primary_metric: NDCG@20
  ks: [1, 5, 10, 20, 50]
wandb:
  project: bpatmp-recsys
  run_name: crgcn-bpatmp
  artifact_name: crgcn-bpatmp-ckpt
"""

yaml_path = '/content/data/training.yaml'
if os.path.isfile(yaml_path):
    with open(yaml_path) as f:
        CFG = yaml.safe_load(f)
else:
    CFG = yaml.safe_load(io.StringIO(YAML_TEXT))

CFG.setdefault('training', {})
T = CFG['training']
T.setdefault('save_dir', '/content/checkpoints-crgcn')
T.setdefault('num_neg', 1)
T.setdefault('max_view_triplets', 3000000)
T.setdefault('max_cart_triplets', 3868050)
T.setdefault('max_purchase_triplets', 0)
T.setdefault('eval_batch_size', 1024)

CFG.setdefault('model', {})
if CFG['model'].get('embed_dim', 64) > 64:
    print('[adjust] embed_dim', CFG['model']['embed_dim'], '-> 64')
    CFG['model']['embed_dim'] = 64
if CFG['model'].get('n_layers', 1) > 2:
    print('[adjust] n_layers', CFG['model']['n_layers'], '-> 1')
    CFG['model']['n_layers'] = 1
if CFG['training'].get('batch_size', 2048) > 2048:
    CFG['training']['batch_size'] = 2048
if CFG['training'].get('max_view_triplets', 0) > 3000000:
    CFG['training']['max_view_triplets'] = 3000000
if CFG['training'].get('max_cart_triplets', 0) > 3868050:
    CFG['training']['max_cart_triplets'] = 3868050

os.makedirs(CFG['training']['save_dir'], exist_ok=True)
print(yaml.safe_dump(CFG, sort_keys=False))

data:
  data_dir: /content/data/
  node_counts:
    product: 29892
    user: 203063
model:
  embed_dim: 64
  n_layers: 1
  dropout: 0.4
training:
  batch_size: 2048
  eval_batch_size: 1024
  epochs: 40
  lr: 0.001
  l2_lambda: 0.0001
  seed: 42
  eval_every: 1
  patience: 5
  num_neg: 1
  device: cuda
  save_dir: /content/checkpoints-crgcn
  max_view_triplets: 3000000
  max_cart_triplets: 3868050
  max_purchase_triplets: 0
evaluation:
  primary_metric: NDCG@20
  ks:
  - 1
  - 5
  - 10
  - 20
  - 50
wandb:
  project: bpatmp-recsys
  run_name: crgcn-bpatmp
  artifact_name: crgcn-bpatmp-ckpt



## 3b. Khởi tạo Weights & Biases

In [4]:
!pip install -q wandb
import wandb
wandb.login(key="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")

WB_CFG = CFG.get('wandb', {}) or {}
run = wandb.init(
    project=WB_CFG.get('project', 'bpatmp-recsys'),
    entity=WB_CFG.get('entity', None),
    name=WB_CFG.get('run_name', 'crgcn'),
    config=CFG,
    reinit=True,
)
ARTIFACT_NAME = WB_CFG.get('artifact_name', 'crgcn-ckpt')
print('wandb run:', run.url)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: nguyenmaiductrong37 (nguyenmaiductrong37-h-c-vi-n-c-ng-ngh-b-u-ch-nh-vi-n-th-ng) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


wandb run: https://wandb.ai/nguyenmaiductrong37-h-c-vi-n-c-ng-ngh-b-u-ch-nh-vi-n-th-ng/bpatmp-recsys/runs/1j6wflwa


## 4. Load BPATMP data → adapter cho CRGCN

CRGCN dùng index 1-based (padding_idx=0); BPATMP dùng 0-based → shift `+1`.

In [5]:
import numpy as np, pickle, torch, json, os
from types import SimpleNamespace

DATA = CFG['data']['data_dir']
N_USERS = CFG['data']['node_counts']['user']
N_ITEMS = CFG['data']['node_counts']['product']

BEHAVIORS = ['view', 'cart', 'purchase']
CAPS = {
    'view':     CFG['training'].get('max_view_triplets', 1_500_000),
    'cart':     CFG['training'].get('max_cart_triplets', 1_500_000),
    'purchase': CFG['training'].get('max_purchase_triplets', 0),
}
RNG = np.random.default_rng(CFG['training']['seed'])

def load_edges(name):
    src = np.load(os.path.join(DATA, f'{name}_train_src.npy')).astype(np.int64)
    dst = np.load(os.path.join(DATA, f'{name}_train_dst.npy')).astype(np.int64)
    cap = CAPS.get(name, 0)
    if cap and len(src) > cap:
        sel = RNG.choice(len(src), size=cap, replace=False)
        src, dst = src[sel], dst[sel]
        print(f'  {name}: subsampled to {cap:,} edges')
    return src, dst

edges = {b: load_edges(b) for b in BEHAVIORS}
for b, (s, d) in edges.items():
    print(f'{b}: edges={len(s):,}  u_max={s.max()}  i_max={d.max()}')

with open(os.path.join(DATA, 'train_mask_purchase_only.pkl'), 'rb') as f:
    TRAIN_MASK = pickle.load(f)
with open(os.path.join(DATA, 'val_ground_truth.pkl'), 'rb') as f:
    VAL_GT = pickle.load(f)
with open(os.path.join(DATA, 'test_ground_truth.pkl'), 'rb') as f:
    TEST_GT = pickle.load(f)
CAND = np.load(os.path.join(DATA, 'candidate_item_idx.npy')).astype(np.int64)
print(f'val users={len(VAL_GT):,}  test users={len(TEST_GT):,}  candidates={len(CAND):,}')

  view: subsampled to 3,000,000 edges
view: edges=3,000,000  u_max=203061  i_max=29890
cart: edges=3,868,050  u_max=203061  i_max=29891
purchase: edges=2,436,760  u_max=203062  i_max=29891
val users=47,996  test users=23,536  candidates=29,892


In [6]:
class BPATMPDataset:
    def __init__(self, n_users, n_items, edges_dict, behaviors):
        self.user_count = n_users
        self.item_count = n_items
        self.behaviors = behaviors
        self.edge_index = {}
        self.behavior_dict = {b: {} for b in behaviors}
        self.behavior_dict['all'] = {}
        self.user_behaviour_degree = []
        for b in behaviors:
            src, dst = edges_dict[b]
            u = torch.from_numpy(src + 1).long()
            i = torch.from_numpy(dst + 1).long()
            deg = torch.zeros(n_users + 1, dtype=torch.float32)
            deg.scatter_add_(0, u, torch.ones_like(u, dtype=torch.float32))
            self.user_behaviour_degree.append(deg.view(-1, 1))
            col = i + (n_users + 1)
            row_all = torch.cat([u, col])
            col_all = torch.cat([col, u])
            self.edge_index[b] = torch.stack([row_all, col_all])
            bd = {}
            for uu, ii in zip(src.tolist(), dst.tolist()):
                bd.setdefault(uu + 1, []).append(ii + 1)
            self.behavior_dict[b] = bd
        all_d = {}
        for b in behaviors:
            for u, lst in self.behavior_dict[b].items():
                all_d.setdefault(u, set()).update(lst)
        self.behavior_dict['all'] = {u: np.array(sorted(s), dtype=np.int64) for u, s in all_d.items()}
        self.user_behaviour_degree = torch.cat(self.user_behaviour_degree, dim=1)

dataset = BPATMPDataset(N_USERS, N_ITEMS, edges, BEHAVIORS)
print('Built dataset. edge_index keys:', list(dataset.edge_index.keys()))

Built dataset. edge_index keys: ['view', 'cart', 'purchase']


## 5. Khởi tạo model CRGCN

In [7]:
import random, numpy as np, torch
SEED = CFG['training']['seed']
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

from model_cascade import CRGCN

device = CFG['training']['device'] if torch.cuda.is_available() else 'cpu'
args = SimpleNamespace(
    device=device,
    layers=[CFG['model']['n_layers']] * len(BEHAVIORS),
    node_dropout=CFG['model']['dropout'],
    message_dropout=CFG['model']['dropout'],
    embedding_size=CFG['model']['embed_dim'],
    behaviors=BEHAVIORS,
    reg_weight=CFG['training']['l2_lambda'],
    model_path=CFG['training']['save_dir'],
    check_point='',
    if_load_model=False,
)
model = CRGCN(args, dataset).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'model on {device} | params = {n_params/1e6:.2f}M')

model on cuda | params = 14.91M


## 6. Evaluator (port từ `evaluator.py`)
Full-rank, tiled, mask exclusion bằng `train_mask_purchase_only`, HR@k & NDCG@k.

In [8]:
@torch.no_grad()
def evaluate(model, gt_dict, exclude_dict, ks=(1,5,10,20,50), user_batch=512, item_tile=16384):
    model.eval()
    model.storage_all_embeddings = None
    max_k = max(ks)
    dev = next(model.parameters()).device
    dtype = torch.float16 if dev.type == 'cuda' else torch.float32

    model.storage_all_embeddings = model.gcn_propagate()
    last_b = model.behaviors[-1]
    u_all, i_all = torch.split(
        model.storage_all_embeddings[last_b],
        [model.n_users + 1, model.n_items + 1],
    )
    item_embs = i_all[1:N_ITEMS + 1].to(dtype)
    user_all = u_all.to(dtype)

    eval_uids = np.array(sorted(gt_dict.keys()), dtype=np.int64)
    n_eval = len(eval_uids)

    gt_lists = [np.asarray(gt_dict[u], dtype=np.int64).reshape(-1) for u in eval_uids]
    max_pos = max(len(g) for g in gt_lists)
    gt_pad = torch.full((n_eval, max_pos), -1, dtype=torch.long, device=dev)
    gt_cnt = torch.zeros(n_eval, dtype=torch.long, device=dev)
    for r, g in enumerate(gt_lists):
        gt_cnt[r] = len(g)
        gt_pad[r, :len(g)] = torch.as_tensor(g, device=dev)

    uid2pos = {int(u): i for i, u in enumerate(eval_uids.tolist())}
    er, ec = [], []
    for u, items in exclude_dict.items():
        p = uid2pos.get(int(u))
        if p is None or len(items) == 0: continue
        items = np.asarray(items, dtype=np.int64).reshape(-1)
        er.extend([p] * len(items)); ec.extend(items.tolist())
    er = torch.as_tensor(er, dtype=torch.long, device=dev)
    ec = torch.as_tensor(ec, dtype=torch.long, device=dev)

    ndcg_w = 1.0 / torch.log2(torch.arange(1, max_k + 1, device=dev).float() + 1.0)
    sums = {f'{m}@{k}': 0.0 for m in ('HR', 'NDCG') for k in ks}

    for us in range(0, n_eval, user_batch):
        ue = min(us + user_batch, n_eval)
        B = ue - us
        uids_b = torch.as_tensor(eval_uids[us:ue] + 1, dtype=torch.long, device=dev)
        u_emb = user_all[uids_b]
        top_v = torch.full((B, max_k), float('-inf'), device=dev, dtype=dtype)
        top_i = torch.full((B, max_k), -1, device=dev, dtype=torch.long)
        inb = (er >= us) & (er < ue)
        sr, sc = er[inb] - us, ec[inb]
        n_items = item_embs.size(0)
        for ts in range(0, n_items, item_tile):
            te = min(ts + item_tile, n_items)
            tile = item_embs[ts:te]
            scores = u_emb @ tile.T
            tmask = (sc >= ts) & (sc < te)
            if tmask.any():
                scores[sr[tmask], sc[tmask] - ts] = float('-inf')
            k = min(max_k, scores.size(1))
            tv, ti = scores.topk(k, dim=-1)
            ti = ti + ts
            mv = torch.cat([top_v, tv], dim=-1)
            mi = torch.cat([top_i, ti], dim=-1)
            sel = mv.topk(max_k, dim=-1).indices
            top_v = mv.gather(1, sel); top_i = mi.gather(1, sel)
        gtb = gt_pad[us:ue]; cnb = gt_cnt[us:ue]
        hits = (top_i.unsqueeze(-1) == gtb.unsqueeze(1)).any(dim=-1)
        for k in ks:
            sums[f'HR@{k}'] += hits[:, :k].any(dim=-1).float().sum().item()
            dcg = (hits[:, :k].float() * ndcg_w[:k]).sum(dim=-1)
            idl = torch.minimum(cnb, torch.full_like(cnb, k))
            idcg = torch.zeros_like(dcg)
            for v in idl.unique():
                mm = idl == v
                if int(v) > 0: idcg[mm] = ndcg_w[:int(v)].sum()
            sums[f'NDCG@{k}'] += (dcg / idcg.clamp_min(1e-12)).sum().item()
    model.storage_all_embeddings = None
    return {k: v / n_eval for k, v in sums.items()}

print('evaluate() defined')

evaluate() defined


## 7. Training loop

In [9]:
from torch.utils.data import Dataset, DataLoader
import time, gc

class BehaviorTripletDataset(Dataset):
    def __init__(self, dataset, behaviors, n_items):
        self.bd = dataset.behavior_dict
        self.behaviors = behaviors
        self.n_users = dataset.user_count
        self.n_items = n_items
    def __len__(self): return self.n_users
    def __getitem__(self, idx):
        u = idx + 1
        out = np.zeros((len(self.behaviors), 3), dtype=np.int64)
        all_pos = self.bd['all'].get(u)
        for k, b in enumerate(self.behaviors):
            lst = self.bd[b].get(u)
            if not lst: continue
            pos = lst[np.random.randint(len(lst))]
            neg = np.random.randint(1, self.n_items + 1)
            if all_pos is not None and len(all_pos) > 0:
                for _ in range(5):
                    if not np.isin(neg, all_pos): break
                    neg = np.random.randint(1, self.n_items + 1)
            out[k] = [u, pos, neg]
        return out

train_ds = BehaviorTripletDataset(dataset, BEHAVIORS, N_ITEMS)
train_loader = DataLoader(train_ds, batch_size=CFG['training']['batch_size'],
                          shuffle=True, num_workers=2, drop_last=True)

optim = torch.optim.Adam(model.parameters(), lr=CFG['training']['lr'])

best_metric = -1.0; best_epoch = -1
patience = CFG['training']['patience']; stale = 0
ks = CFG['evaluation']['ks']; primary = CFG['evaluation']['primary_metric']
save_dir = CFG['training']['save_dir']
best_ckpt_path = os.path.join(save_dir, 'best.pt')

def upload_ckpt_to_wandb(path, epoch, metric_value, aliases=None):
    art = wandb.Artifact(ARTIFACT_NAME, type='model',
                         metadata={'epoch': epoch, primary: metric_value})
    art.add_file(path)
    run.log_artifact(art, aliases=aliases or ['latest'])

def free_vram():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

for epoch in range(1, CFG['training']['epochs'] + 1):
    model.train(); t0 = time.time(); ep_loss = 0.0; nb = 0
    for batch in train_loader:
        batch = batch.to(device)
        loss = model(batch)
        optim.zero_grad(); loss.backward(); optim.step()
        ep_loss += float(loss.item()); nb += 1
        del batch, loss
    free_vram()
    avg_loss = ep_loss / max(nb, 1); dt = time.time() - t0
    print(f'epoch {epoch:03d} | loss {avg_loss:.4f} | {dt:.1f}s')
    log_payload = {'epoch': epoch, 'train/loss': avg_loss, 'train/epoch_seconds': dt}

    if epoch % CFG['training']['eval_every'] == 0:
        m = evaluate(model, VAL_GT, TRAIN_MASK, ks=ks,
                     user_batch=CFG['training']['eval_batch_size'])
        free_vram()
        print('  val:', {k: round(v, 4) for k, v in m.items()})
        for k, v in m.items(): log_payload[f'val/{k}'] = v
        cur = m[primary]
        if cur > best_metric:
            best_metric = cur; best_epoch = epoch; stale = 0
            torch.save(model.state_dict(), best_ckpt_path)
            upload_ckpt_to_wandb(best_ckpt_path, epoch, cur, aliases=['best', 'latest'])
            print(f'new best {primary}={cur:.4f} (saved + uploaded)')
            log_payload['val/best_metric'] = cur
            log_payload['val/best_epoch'] = epoch
        else:
            stale += 1
            wandb.log(log_payload, step=epoch)
            if stale >= patience:
                print(f'  early stop at epoch {epoch} (best epoch {best_epoch})')
                break
            continue
    wandb.log(log_payload, step=epoch)

print(f'\nDone. best {primary}={best_metric:.4f} @ epoch {best_epoch}')

epoch 001 | loss 0.5291 | 23.8s
  val: {'HR@1': 0.038, 'HR@5': 0.1287, 'HR@10': 0.1901, 'HR@20': 0.2681, 'HR@50': 0.3856, 'NDCG@1': 0.038, 'NDCG@5': 0.054, 'NDCG@10': 0.0671, 'NDCG@20': 0.0833, 'NDCG@50': 0.1066}
new best NDCG@20=0.0833 (saved + uploaded)
epoch 002 | loss 0.2923 | 22.3s
  val: {'HR@1': 0.0433, 'HR@5': 0.1324, 'HR@10': 0.1943, 'HR@20': 0.2693, 'HR@50': 0.3809, 'NDCG@1': 0.0433, 'NDCG@5': 0.0575, 'NDCG@10': 0.0712, 'NDCG@20': 0.087, 'NDCG@50': 0.1094}
new best NDCG@20=0.0870 (saved + uploaded)
epoch 003 | loss 0.2532 | 22.7s
  val: {'HR@1': 0.0421, 'HR@5': 0.1313, 'HR@10': 0.1924, 'HR@20': 0.2664, 'HR@50': 0.3713, 'NDCG@1': 0.0421, 'NDCG@5': 0.0575, 'NDCG@10': 0.0711, 'NDCG@20': 0.0872, 'NDCG@50': 0.1078}
new best NDCG@20=0.0872 (saved + uploaded)
epoch 004 | loss 0.2436 | 22.5s
  val: {'HR@1': 0.0384, 'HR@5': 0.1201, 'HR@10': 0.1792, 'HR@20': 0.2484, 'HR@50': 0.3541, 'NDCG@1': 0.0384, 'NDCG@5': 0.0528, 'NDCG@10': 0.0657, 'NDCG@20': 0.0805, 'NDCG@50': 0.1013}
epoch 005 |

## 8. Đánh giá trên test set với checkpoint tốt nhất

In [10]:
ckpt = os.path.join(save_dir, 'best.pt')
if os.path.isfile(ckpt):
    model.load_state_dict(torch.load(ckpt, map_location=device))
    print('loaded', ckpt)
m_test = evaluate(model, TEST_GT, TRAIN_MASK, ks=CFG['evaluation']['ks'],
                  user_batch=CFG['training']['eval_batch_size'])
print('TEST metrics:')
for k in CFG['evaluation']['ks']:
    print(f'  HR@{k:>3} = {m_test[f"HR@{k}"]:.4f}   NDCG@{k:>3} = {m_test[f"NDCG@{k}"]:.4f}')

wandb.log({f'test/{k}': v for k, v in m_test.items()})
wandb.summary.update({f'test/{k}': v for k, v in m_test.items()})
wandb.summary['best_val_epoch'] = best_epoch
wandb.summary[f'best_val_{primary}'] = best_metric

final_art = wandb.Artifact(ARTIFACT_NAME, type='model',
                           metadata={'best_epoch': best_epoch,
                                     f'val_{primary}': best_metric,
                                     **{f'test_{k}': v for k, v in m_test.items()}})
final_art.add_file(ckpt)
run.log_artifact(final_art, aliases=['final'])
wandb.finish()

loaded /content/checkpoints-crgcn/best.pt
TEST metrics:
  HR@  1 = 0.0226   NDCG@  1 = 0.0226
  HR@  5 = 0.0789   NDCG@  5 = 0.0365
  HR@ 10 = 0.1250   NDCG@ 10 = 0.0478
  HR@ 20 = 0.1809   NDCG@ 20 = 0.0599
  HR@ 50 = 0.2727   NDCG@ 50 = 0.0771


epoch,▁▂▃▄▅▆▇█
test/HR@1,▁
test/HR@10,▁
test/HR@20,▁
test/HR@5,▁
test/HR@50,▁
test/NDCG@1,▁
test/NDCG@10,▁
test/NDCG@20,▁
test/NDCG@5,▁
+15,...
